In [9]:
import xgboost as xgb
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
import src.train_utils as T
import numpy as np

In [12]:
df = pd.read_csv('../datasets/base.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [13]:
train_df.loc[:, 'wind_speed_t'] = np.sqrt(
    train_df['u_wind_10m_t'] ** 2 + train_df['v_wind_10m_t'] ** 2
)

/var/folders/7_/y4vpj6ln7b90djzq_hscbgyr0000gn/T/ipykernel_1737/173462167.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df.loc[:, 'wind_speed_t'] = np.sqrt(


In [14]:
base = []

feature_experiments = [
    ('persistence', base),
]

model=DummyRegressor(strategy='constant', constant=0)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)
print(results.sort_values('weighted_rmse'))

Running experiment: persistence
Fold 0
RMSE: 1.7746019244355444
Fold 1
RMSE: 6.269512050039968
Fold 2
RMSE: 4.223291823935145
Fold 3
RMSE: 8.669010890470068
Fold 4
RMSE: 14.754026021929128
Fold 5
RMSE: 4.4301957827731835
Fold 6
RMSE: 0.9896191160986055
Fold 7
RMSE: 1.0655872014374173
Fold 8
RMSE: 1.653758701602382
Fold 9
RMSE: 4.126343254845557
    experiment  n_features  unweighted_rmse  weighted_rmse
0  persistence           0         4.795595        6.63755


In [15]:
base = ['pm25_t', 'delta_pm25_t']

params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('base', base),
    ('base + u-v_wind', base + ['u_wind_10m_t', 'v_wind_10m_t']),
    ('base + wind_speed', base + ['wind_speed_t']),
    ('base + dew_temp_2m_t', base + ['dew_temp_2m_t']),
    ('base + temp_2m_t', base + ['temp_2m_t']),
    ('base + surf_pressure_t', base + ['surf_pressure_t']),
    ('base + precip_sum_t', base + ['precip_sum_t']),
    ('base + frp_t', base + ['frp_t']),
    ('base + elevation_t', base + ['elevation_t']),
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7978426125740559
Fold 1
RMSE: 5.862707907283073
Fold 2
RMSE: 4.211544698843767
Fold 3
RMSE: 8.561094818282895
Fold 4
RMSE: 14.314654705440923
Fold 5
RMSE: 4.521556747185893
Fold 6
RMSE: 1.0342208505385009
Fold 7
RMSE: 1.1294038099794623
Fold 8
RMSE: 1.6936859406106886
Fold 9
RMSE: 4.123966813520194
Running experiment: base + u-v_wind
Fold 0
RMSE: 1.7947847898488134
Fold 1
RMSE: 5.611102182475015
Fold 2
RMSE: 4.199158409550276
Fold 3
RMSE: 8.55829387980443
Fold 4
RMSE: 14.279265970501182
Fold 5
RMSE: 4.565510552952192
Fold 6
RMSE: 1.053661283078711
Fold 7
RMSE: 1.1443578802383043
Fold 8
RMSE: 1.6913395077857716
Fold 9
RMSE: 4.087956591734257
Running experiment: base + wind_speed
Fold 0
RMSE: 1.808397948429805
Fold 1
RMSE: 5.907406464022214
Fold 2
RMSE: 4.210897814123812
Fold 3
RMSE: 8.540523931095388
Fold 4
RMSE: 14.5208346910114
Fold 5
RMSE: 4.533044519012801
Fold 6
RMSE: 1.0389934628565083
Fold 7
RMSE: 1.1456957855663876
Fold 8
RMSE: 1.702018438

In [16]:
base = ['row', 'col', 'pm25_t', 'u_wind_10m_t', 'v_wind_10m_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t']

params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('without elevation', base),
    ('with elevation', base + ['elevation_t']),
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: without elevation
Fold 0
RMSE: 1.7536182258531305
Fold 1
RMSE: 5.71978637639383
Fold 2
RMSE: 4.338275977109895
Fold 3
RMSE: 8.350090827552068
Fold 4
RMSE: 13.4465363690072
Fold 5
RMSE: 4.035674209473935
Fold 6
RMSE: 0.9910860908056804
Fold 7
RMSE: 1.0826023169731047
Fold 8
RMSE: 1.6478248488217369
Fold 9
RMSE: 4.082657891630055
Running experiment: with elevation
Fold 0
RMSE: 1.7538971199717432
Fold 1
RMSE: 5.646030655159194
Fold 2
RMSE: 4.340488967361895
Fold 3
RMSE: 8.3187428002335
Fold 4
RMSE: 13.417024429553564
Fold 5
RMSE: 4.06069866533777
Fold 6
RMSE: 0.9935626626820864
Fold 7
RMSE: 1.0876721615145977
Fold 8
RMSE: 1.6509902340005804
Fold 9
RMSE: 4.0841524497541695
          experiment  n_features  unweighted_rmse  weighted_rmse
1     with elevation          11         4.535326       6.227251
0  without elevation          10         4.544815       6.241669


In [17]:
base = ['row', 'col', 'pm25_t',
       'dew_temp_2m_t', 'temp_2m_t', 'surf_pressure_t', 'precip_sum_t',
       'frp_t', 'elevation_t']

params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    ('u-v_wind', base + ['u_wind_10m_t', 'v_wind_10m_t']),
    ('wind speed', base + ['wind_speed_t']),
    ('both', base + ['u_wind_10m_t', 'v_wind_10m_t', 'wind_speed_t'])
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=train_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: u-v_wind
Fold 0
RMSE: 1.7539111636787381
Fold 1
RMSE: 5.654649753585336
Fold 2
RMSE: 4.339160638460471
Fold 3
RMSE: 8.33031341413366
Fold 4
RMSE: 13.497309315455116
Fold 5
RMSE: 4.043083556017283
Fold 6
RMSE: 0.9944917088728068
Fold 7
RMSE: 1.0819055466394374
Fold 8
RMSE: 1.6436543814638418
Fold 9
RMSE: 4.074383102953968
Running experiment: wind speed
Fold 0
RMSE: 1.7500130877050788
Fold 1
RMSE: 5.737495881542018
Fold 2
RMSE: 4.3886459088416805
Fold 3
RMSE: 8.347171580715388
Fold 4
RMSE: 13.49537783997648
Fold 5
RMSE: 4.049995824735535
Fold 6
RMSE: 0.9875590205873189
Fold 7
RMSE: 1.0789788648862766
Fold 8
RMSE: 1.6476606477953275
Fold 9
RMSE: 4.048676864087385
Running experiment: both
Fold 0
RMSE: 1.7587021999136314
Fold 1
RMSE: 5.717912575100526
Fold 2
RMSE: 4.36906121165751
Fold 3
RMSE: 8.298923786957518
Fold 4
RMSE: 13.508472142656208
Fold 5
RMSE: 4.138589815844519
Fold 6
RMSE: 0.9872428179851106
Fold 7
RMSE: 1.080845786257707
Fold 8
RMSE: 1.6448541627875513
Fold